In [0]:
%pip install python-dotenv -q
%restart_python

In [0]:
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os


load_dotenv(override=True)
CLIENT_ID = os.environ.get("CLIENT_ID") # created with UI
CLIENT_SECRET = os.environ.get("CLIENT_SECRET") # created with UI
dbutils.widgets.text("secret_scope", "demo-scope") # pre-created before
SECRET_SCOPE = dbutils.widgets.get("secret_scope")

workspace = WorkspaceClient()
workspace.secrets.put_secret(SECRET_SCOPE, "road_traffic_experiment_client_id", string_value =CLIENT_ID)
workspace.secrets.put_secret(SECRET_SCOPE, "road_traffic_experiment_client_secret", string_value =CLIENT_SECRET)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.ml import ExperimentAccessControlRequest
from databricks.sdk.service.iam import PermissionLevel
import mlflow



# MLflow experiment to capture traces
experiment = mlflow.set_experiment("/Workspace/Shared/road-traffic-accident-analysis-agent-experiment")
experiment_id = experiment.experiment_id

# Fetch the service principal client_id from secret scope
client_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="road_traffic_experiment_client_id")

# Set permissions for the SP which will later write the traces from the serving endpoint
workspace = WorkspaceClient()
# Set CAN_EDIT permissions for the service principal
workspace.experiments.set_permissions(
    experiment_id=experiment_id,
    access_control_list=[
        ExperimentAccessControlRequest(
            service_principal_name=client_id,
            permission_level=PermissionLevel.CAN_EDIT
        )
    ]
)

print(f"✓ CAN_EDIT permissions granted to SPN {client_id[:8]}... for experiment: {experiment_id}")